In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from medicalDataLoader_Enes import MedicalImageDataset
from utils_Enes import device, evaluate, show_predictions, show_confusion
from UNet_Base_Enes import UNet

DATA_DIR = '../../Data/'
RUNS = {
    '1A supervisé': '../baseline/models/final_dice',
    'Pseudo-labels': 'models/pseudo_labels',
    'Mean Teacher': 'models/mean_teacher',
}
MODELS = {
    '1A supervisé': '../baseline/models/final_dice/best.pth',
    'Pseudo-labels': 'models/pseudo_labels/best.pth',
    'Mean Teacher': 'models/mean_teacher/last.pth',
}
histories = {name: np.load(d + '/history.npy', allow_pickle=True).item() for name, d in RUNS.items()}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in histories.items():
    dice = np.array(h['val_dice'], dtype=float).mean(axis=1)
    axes[0].plot(dice, label='{} ({:.3f})'.format(name, dice[-10:].mean()))
    axes[1].plot(np.nanmean(np.array(h['val_hd'], dtype=float), axis=1), label=name)
axes[0].set_ylim(0.6, 0.9)
axes[0].set_title('Dice moyen de validation')
axes[1].set_ylim(0, 25)
axes[1].set_title('Hausdorff moyen de validation (px)')
axes[0].legend(title='Moyenne des 10 dernières époques')
axes[1].legend()
plt.savefig('figures/courbes_validation.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k, c in enumerate(['VD', 'Myo', 'VG']):
    for name, h in histories.items():
        axes[k].plot(np.array(h['val_dice'], dtype=float)[:, k], label=name)
    axes[k].set_ylim(0.5, 1)
    axes[k].set_title('Dice de validation : ' + c)
axes[0].legend()
plt.savefig('figures/dice_par_classe.png', dpi=150)
plt.show()

In [ ]:
h = histories['Mean Teacher']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(np.array(h['val_dice'], dtype=float).mean(axis=1), label='Professeur')
axes[0].plot(np.array(h['student_dice'], dtype=float).mean(axis=1), label='Élève')
axes[0].set_title('Mean Teacher : professeur et élève')
axes[0].legend()
axes[1].plot(h['train_loss'], label='Loss supervisée')
axes[1].plot(h['cons_loss'], label='Loss de cohérence')
axes[1].set_yscale('log')
axes[1].legend()
plt.savefig('figures/mean_teacher.png', dpi=150)
plt.show()

In [ ]:
val_set = MedicalImageDataset('val', DATA_DIR)
val_loader = DataLoader(val_set, batch_size=8)
nets = {}
for name, path in MODELS.items():
    nets[name] = UNet(num_classes=4).to(device)
    nets[name].load_state_dict(torch.load(path, map_location=device))
    last10 = np.array(histories[name]['val_dice'], dtype=float)[-10:].mean()
    dice, hd = evaluate(nets[name], val_loader, post_process=True)
    print('{:<15} 10 dern. {:.3f} | Dice VD {:.3f} Myo {:.3f} VG {:.3f} (moy. {:.3f}) | HD VD {:.1f} Myo {:.1f} VG {:.1f}'.format(
        name, last10, *dice, np.mean(dice), *hd))

In [ ]:
for name in ['1A supervisé', 'Mean Teacher']:
    print(name)
    show_confusion(nets[name], val_loader, post_process=True)
    show_predictions(nets[name], val_set, [5, 20, 40, 60], post_process=True)